<a href="https://colab.research.google.com/github/gordon921212/Baseball-Project/blob/main/baseball.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pybaseball

In [4]:
from pybaseball import statcast

df_pitch_by_pitch = statcast(start_dt="2025-03-01", end_dt="2025-11-30")


This is a large query, it may take a moment to complete


/usr/local/lib/python3.12/dist-packages/pybaseball/statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)


Skipping offseason dates
Skipping offseason dates


  0%|          | 0/246 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/usr/local/lib/python3.12/dist-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/usr/local/lib/python3.12/dist-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = 

In [6]:
df_in_play= df_pitch_by_pitch[df_pitch_by_pitch['description'] == 'hit_into_play']
df_in_play[['launch_speed','launch_angle','events','hit_distance_sc']]

,launch_speed,launch_angle,events,hit_distance_sc
44,71.9,-24,grounded_into_double_play,5
72,11.9,2,sac_bunt,7
79,89.8,21,double,291
210,93.8,0,field_out,47
292,104.6,39,home_run,366
...,...,...,...,...
3296,<NA>,<NA>,field_out,<NA>
3874,<NA>,<NA>,field_out,<NA>
4750,<NA>,<NA>,field_out,<NA>
2063,<NA>,<NA>,field_out,<NA>


In [8]:
print(df_in_play['events'].unique())
print("各 events 種類的出現次數：")
print(df_in_play['events'].value_counts())

['grounded_into_double_play' 'sac_bunt' 'double' 'field_out' 'home_run'
 'force_out' 'single' 'sac_fly' 'double_play' 'field_error' 'triple'
 'fielders_choice_out' 'fielders_choice' 'sac_fly_double_play'
 'catcher_interf' 'triple_play']
各 events 種類的出現次數：
events
field_out                    79949
single                       28272
double                        8408
home_run                      6070
force_out                     3652
grounded_into_double_play     3405
sac_fly                       1397
field_error                   1112
triple                         690
sac_bunt                       596
double_play                    412
fielders_choice                412
fielders_choice_out            349
sac_fly_double_play             19
catcher_interf                  13
triple_play                      3
Name: count, dtype: int64


In [9]:
import pandas as pd

print("開始進行精確的標籤編碼...")

# 1. 剔除雜訊：把「捕手妨礙打擊」的無效數據直接刪掉
df_model = df_in_play[df_in_play['events'] != 'catcher_interf'].copy()

# 2. 定義安打的種類
hit_events = ['single', 'double', 'triple', 'home_run']

# 3. 標籤編碼 (Label Encoding)
# 只要是 hit_events 就是 1，其餘 11 種出局或失誤全部歸為 0
df_model['is_hit'] = df_model['events'].apply(lambda x: 1 if x in hit_events else 0)

# 4. 留下我們需要的欄位 (特徵 X 與目標 Y)
df_model = df_model[['launch_speed', 'launch_angle', 'is_hit']]

# 5. 清除雷達當機沒測到初速/仰角的缺失值
df_model = df_model.dropna()

print("========================================")
print(f"✅ 資料清理完成！共保留 {len(df_model)} 筆有效特徵資料。")
print("========================================")

# 檢查一下 0 和 1 的分佈
print(df_model['is_hit'].value_counts(normalize=True) * 100)

開始進行精確的標籤編碼...
✅ 資料清理完成！共保留 131637 筆有效特徵資料。
is_hit
0    67.77046
1    32.22954
Name: proportion, dtype: float64


In [10]:
from sklearn.model_selection import train_test_split

print("開始切分資料集 (Train-Test Split)...")

# 1. 將特徵 (X) 與目標 (y) 分開
# X 是模型學習的條件：擊球初速與仰角
X = df_model[['launch_speed', 'launch_angle']]
# y 是模型要預測的答案：是否為安打 (0 或 1)
y = df_model['is_hit']

# 2. 執行切分 (80% 訓練集, 20% 測試集)
# stratify=y 確保切分後的訓練集和測試集，安打與出局的比例和原始資料一模一樣，不會剛好切到全都是出局的極端考卷。
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 保留 20% 的資料作為最後的「期末考」
    random_state=42,     # 設定固定的亂數種子，確保你每次跑這段程式碼，切出來的結果都一樣，方便 Debug
    stratify=y           # 資料不平衡時必備的參數
)

print("========================================")
print("✅ 資料切分完成！")
print(f"總資料筆數: {len(df_model)}")
print(f"➔ 訓練集 (X_train) 用於訓練模型: {len(X_train)} 筆 (80%)")
print(f"➔ 測試集 (X_test) 用於評估準確度: {len(X_test)} 筆 (20%)")
print("========================================")

# 驗證 stratify 的效果 (兩邊的安打比例應該要幾乎完全相同)
print("\n[訓練集 y_train 的安打與出局比例]")
print((y_train.value_counts(normalize=True) * 100).round(2).astype(str) + '%')

print("\n[測試集 y_test 的安打與出局比例]")
print((y_test.value_counts(normalize=True) * 100).round(2).astype(str) + '%')

開始切分資料集 (Train-Test Split)...
✅ 資料切分完成！
總資料筆數: 131637
➔ 訓練集 (X_train) 用於訓練模型: 105309 筆 (80%)
➔ 測試集 (X_test) 用於評估準確度: 26328 筆 (20%)

[訓練集 y_train 的安打與出局比例]
is_hit
0    67.77%
1    32.23%
Name: proportion, dtype: object

[測試集 y_test 的安打與出局比例]
is_hit
0    67.77%
1    32.23%
Name: proportion, dtype: object


In [16]:
from sklearn.ensemble import RandomForestClassifier

# 建立模型時，多加一個 oob_score=True 參數
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    class_weight='balanced',
    oob_score=True,           # 🌟 開啟這項，讓它自動進行袋外驗證！
    random_state=42,
    n_jobs=-1
)

# 直接餵給它完整的訓練集 (X_train, y_train)，不用再切 sub 和 val 了！
rf_model.fit(X_train, y_train)

# 訓練完成後，直接呼叫它的 oob_score_ 屬性
print(f"🌲 隨機森林自動驗證 (OOB Score) 準確率: {rf_model.oob_score_:.4f}")

🌲 隨機森林自動驗證 (OOB Score) 準確率: 0.7718


In [18]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

print("開始進行最終期末考：使用測試集 (Test Set) 評估模型...")

# 1. 讓訓練好的隨機森林模型對測試集進行預測
y_test_pred = rf_model.predict(X_test)          # 預測絕對結果 (0 或 1)
y_test_proba = rf_model.predict_proba(X_test)[:, 1] # 預測安打機率 (xBA)

# 2. 計算最終關鍵指標
final_accuracy = accuracy_score(y_test, y_test_pred)
final_auc = roc_auc_score(y_test, y_test_proba)

print("\n========================================")
print("🎓 期末專案最終測試集 (Test Set) 評估報告")
print("========================================")
print(f"最終準確率 (Final Accuracy): {final_accuracy:.4f}")
print(f"最終 ROC-AUC 分數:           {final_auc:.4f}")

# 3. 印出詳細的分類報告 (包含 Precision, Recall, F1-score)
print("\n[詳細分類報告]")
print(classification_report(y_test, y_test_pred, target_names=['出局 (0)', '安打 (1)']))


開始進行最終期末考：使用測試集 (Test Set) 評估模型...

🎓 期末專案最終測試集 (Test Set) 評估報告
最終準確率 (Final Accuracy): 0.7673
最終 ROC-AUC 分數:           0.8593

[詳細分類報告]
              precision    recall  f1-score   support

      出局 (0)       0.89      0.75      0.81     17843
      安打 (1)       0.61      0.80      0.69      8485

    accuracy                           0.77     26328
   macro avg       0.75      0.78      0.75     26328
weighted avg       0.80      0.77      0.77     26328

